In [1]:
import pandas as pd
df = pd.read_csv("data/dengue_data_with_weather_data.csv")
df.shape

(900, 11)

In [2]:
df.columns.tolist()

['Year',
 'Province',
 'District',
 'Latitude',
 'Longitude',
 'Elevation',
 'Month',
 'Cases',
 'Temp_avg',
 'Precipitation_avg',
 'Humidity_avg']

In [3]:
df.head(10)

,Year,Province,District,Latitude,Longitude,Elevation,Month,Cases,Temp_avg,Precipitation_avg,Humidity_avg
0,2019,Western,Colombo,6.924429,79.907250,4,1,1225,26.914286,0.771429,72.607143
1,2019,Western,Gampaha,7.065026,79.966220,19,1,724,27.417857,0.728571,69.892857
2,2019,Western,Kalutara,6.572935,80.025190,5,1,372,26.112500,1.396429,80.464286
3,2019,Central,Kandy,7.275923,80.626590,499,1,288,22.067857,1.853571,83.428571
4,2019,Central,Matale,7.486819,80.526320,362,1,62,25.592857,0.850000,74.285714
5,2019,Central,Nuwara Eliya,6.994728,80.734180,1865,1,26,14.741071,2.717857,86.571429
6,2019,Southern,Galle,6.080843,80.233730,8,1,132,25.432143,2.953571,85.285714
7,2019,Southern,Hambantota,5.940246,80.550000,6,1,140,25.760714,0.828571,83.357143
8,2019,Southern,Matara,6.151143,81.127815,8,1,219,27.287500,0.710714,71.000000
9,2019,Northern,Jaffna,9.666080,80.034874,8,1,900,25.105357,0.178571,74.892857


In [4]:
print(df.isnull().sum())
print(df.Year.unique())
print(df.District.nunique())
print(df.groupby(['District', 'Year']).size().unique())

Year                 0
Province             0
District             0
Latitude             0
Longitude            0
Elevation            0
Month                0
Cases                0
Temp_avg             1
Precipitation_avg    1
Humidity_avg         1
dtype: int64
[2019 2020 2021]
25
[12]


In [5]:
df[df.isnull().any(axis=1)]

,Year,Province,District,Latitude,Longitude,Elevation,Month,Cases,Temp_avg,Precipitation_avg,Humidity_avg
151,2019,Western,Gampaha,7.065026,79.96622,19,3,511,NaN,NaN,NaN


In [6]:
df[['Temp_avg', 'Precipitation_avg', 'Humidity_avg']] = (
    df.groupby('District')[['Temp_avg', 'Precipitation_avg', 'Humidity_avg']]
    .transform(lambda s: s.fillna(s.mean()))
)

df.isnull().sum()

Year                 0
Province             0
District             0
Latitude             0
Longitude            0
Elevation            0
Month                0
Cases                0
Temp_avg             0
Precipitation_avg    0
Humidity_avg         0
dtype: int64

In [7]:
df.Cases.describe()
print(df.groupby('District').Cases.mean().sort_values(ascending=False))

District
Colombo         1010.444444
Gampaha          712.444444
Kandy            385.166667
Kalutara         362.250000
Jaffna           294.194444
Batticaloa       280.000000
Galle            271.111111
Ratnapura        195.277778
Kurunegala       167.083333
Trincomalee      154.611111
Matara           148.666667
Kegalle          120.777778
Ampara           100.750000
Puttalam          97.500000
Matale            92.500000
Badulla           89.166667
Hambantota        80.111111
Anuradhapura      54.694444
Vavuniya          32.722222
Polonnaruwa       26.000000
Mannar            20.722222
Nuwara Eliya      19.972222
Kilinochchi       15.472222
Moneragala        14.805556
Mulativu          10.000000
Name: Cases, dtype: float64


In [8]:
district_stats = df.groupby('District')['Cases'].agg(['mean', 'std']).rename(columns={'mean': 'd_mean', 'std': 'd_std'})
df = df.merge(district_stats, on='District', how='left')
df['z_score'] = (df['Cases'] - df['d_mean']) / df['d_std']

df[['District', 'Year', 'Month', 'Cases', 'd_mean', 'z_score']].sort_values('z_score', ascending=False).head(10)

,District,Year,Month,Cases,d_mean,z_score
886,Mannar,2021,12,284,20.722222,5.049149
835,Kilinochchi,2019,12,141,15.472222,4.315080
753,Kandy,2019,11,2444,385.166667,4.162248
754,Matale,2019,11,902,92.500000,4.124261
837,Vavuniya,2019,12,306,32.722222,4.024973
41,Trincomalee,2020,1,1455,154.611111,3.894759
834,Jaffna,2019,12,2763,294.194444,3.850181
768,Puttalam,2019,11,624,97.500000,3.820037
40,Ampara,2020,1,682,100.750000,3.758709
759,Jaffna,2019,11,2641,294.194444,3.659918


In [9]:
df['risk_tier'] = df['z_score'].apply(lambda z: 'Elevated' if z > 0.5 else 'Normal')
df['risk_tier'].value_counts()

risk_tier
Normal      742
Elevated    158
Name: count, dtype: int64

In [10]:
df['Cases_lag1'] = df.groupby('District')['Cases'].shift(1)

In [11]:
df[df.District == 'Colombo'][['Year', 'Month', 'Cases', 'Cases_lag1']].head(5)

,Year,Month,Cases,Cases_lag1
0,2019,1,1225,NaN
25,2020,1,1693,1225.0
50,2021,1,208,1693.0
75,2019,2,792,208.0
100,2020,2,767,792.0


In [12]:
df['Temp_avg_lag1'] = df.groupby('District')['Temp_avg'].shift(1)
df['Precipitation_avg_lag1'] = df.groupby('District')['Precipitation_avg'].shift(1)
df['Humidity_avg_lag1'] = df.groupby('District')['Humidity_avg'].shift(1)

In [13]:
df[df.District == 'Colombo'][['Year', 'Month', 'Temp_avg', 'Temp_avg_lag1']].head(5)

,Year,Month,Temp_avg,Temp_avg_lag1
0,2019,1,26.914286,NaN
25,2020,1,26.882143,26.914286
50,2021,1,25.755357,26.882143
75,2019,2,27.185714,25.755357
100,2020,2,28.030357,27.185714


In [14]:
df['Cases_roll3'] = df.groupby('District')['Cases'].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

In [15]:
df[df.District == 'Colombo'][['Year', 'Month', 'Cases', 'Cases_lag1', 'Cases_roll3']].head(6)

,Year,Month,Cases,Cases_lag1,Cases_roll3
0,2019,1,1225,NaN,NaN
25,2020,1,1693,1225.0,1225.000000
50,2021,1,208,1693.0,1459.000000
75,2019,2,792,208.0,1042.000000
100,2020,2,767,792.0,897.666667
125,2021,2,388,767.0,589.000000


In [16]:
df_clean = df.dropna(subset=['Cases_lag1', 'Temp_avg_lag1', 'Precipitation_avg_lag1', 'Humidity_avg_lag1']).copy()
df_clean.shape

(875, 20)

In [17]:
from sklearn.preprocessing import LabelEncoder

le_district = LabelEncoder()
df_clean['district_enc'] = le_district.fit_transform(df_clean['District'])

df_clean[['District', 'district_enc']].drop_duplicates().head(10)

,District,district_enc
25,Colombo,4
26,Gampaha,6
27,Kalutara,9
28,Kandy,10
29,Matale,15
30,Nuwara Eliya,19
31,Galle,5
32,Hambantota,7
33,Matara,16
34,Jaffna,8


In [18]:
feature_cols = [
    'district_enc', 'Month',
    'Cases_lag1', 'Cases_roll3',
    'Temp_avg_lag1', 'Precipitation_avg_lag1', 'Humidity_avg_lag1',
]

X = df_clean[feature_cols]
y = df_clean['risk_tier']

X.shape, y.shape

((875, 7), (875,))

In [19]:
train_mask = df_clean['Year'] < 2021
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=4, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

from sklearn.metrics import classification_report
pred = clf.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    Elevated       0.10      0.79      0.17        14
      Normal       0.98      0.64      0.78       286

    accuracy                           0.65       300
   macro avg       0.54      0.71      0.48       300
weighted avg       0.94      0.65      0.75       300



In [20]:
import joblib
joblib.dump(clf, 'risk_model.joblib')
joblib.dump(le_district, 'district_encoder.joblib')

district_stats.to_csv('district_baselines.csv')